In [0]:
from pyspark.sql.functions import to_date, try_to_date, split, col, element_at, lit, when, length, concat, expr, to_timestamp, concat_ws

In [0]:
silver_bt = spark.table("workspace.default.bronze_bank_transactions")

In [0]:
silver_bt.printSchema()

In [0]:
silver_bt.filter(col("CustomerDOB").isNull()).count()

In [0]:
from pyspark.sql import functions as F

In [0]:
# bad_balance = silver_bt.filter(
#     F.col("CustAccountBalance").isNotNull() &
#     F.expr("try_cast(CustAccountBalance AS DOUBLE)").isNull()
# )

# bad_balance.count()

Checking if <br>

    -CustAccountBalance
    -TransactionAmountINR


are null in general or after typecast becomes null as they are 'a23' and gets casted to null
We first ran F.expr - which basically runs SQL code and returns back pyspark col object
then try_cast for a safe type cast conversion which returns null instead of throwing an error or runtime exception if value cannot be converted.

In [0]:
bad_balance = silver_bt.filter(F.expr("try_cast(CustAccountBalance AS DOUBLE)").isNull())
bad_balance.count()

In [0]:
silver_bt.filter(F.col("CustAccountBalance").isNull()).count()

In [0]:
silver_bt.filter(F.col("TransactionAmountINR").isNull()).count()

In [0]:
silver_bt.filter(F.expr("try_cast(TransactionAmountINR AS DOUBLE)").isNull()).count()

### type casting

In [0]:
# silver_bt = silver_bt.withColumn("CustAccountBalance", F.expr("try_cast(CustAccountBalance AS DOUBLE)"))
silver_bt = silver_bt.withColumn("CustAccuntBalance", F.col("CustAccountBalance").cast("double"))

In [0]:
silver_bt.filter(F.expr("try_cast(CustomerDOB AS DATE)").isNull()).count()

In [0]:
silver_bt.count()

In [0]:
silver_bt.filter(F.col("CustomerDOB").isNull()).count()

In [0]:
silver_bt.select("CustomerDOB").show(10)

In [0]:
silver_bt.withColumn("DOB_parse", try_to_date(F.col("CustomerDOB"), "d/M/yy")).select("CustomerDOB", "DOB_parse").show(10)

In [0]:
temp_df = silver_bt.withColumn("dob_parts", split(col("CustomerDOB"), "/"))
temp_df.select("CustomerDOB", "dob_parts").show(10)

In [0]:
# temp_df = temp_df.select("dob_parts").element_at("dob_parts",3).length()
# temp_df = temp_df.concat(temp_df, "19")

# temp_df = temp_df.withColumn("dob_year_raw", element_at(col("dob_parts"), 3))
# temp_df.select("CustomerDOB", "dob_parts", "dob_year_raw").show(10)

In [0]:
# from pyspark.sql.functions import size

# temp_df.filter(size(col("dob_parts")) != 3).select("CustomerDOB").show(20, truncate=False)


In [0]:
silver_bt.filter(col("CustomerDOB") == 'nan').count()

converted nan value to null

In [0]:
silver_bt = silver_bt.withColumn(
    "CustomerDOB", 
    when(col("CustomerDOB")=='nan', lit(None)).otherwise(col("CustomerDOB")))

In [0]:
silver_bt.filter(col("CustomerDOB").isNull()).count()

In [0]:
temp_df = silver_bt.withColumn("dob_parts", split(col("CustomerDOB"), "/"))

In [0]:
temp_df.select("CustomerDOB", "dob_parts").show(5)

In [0]:
# temp_df.filter(col('CustomerDOB').isNull()).select("CustomerDOB", "dob_parts").show(5)

In [0]:
# # Some checks for Null in customerDOB
# temp_df.count()
# silver_bt.count()
# temp_df.filter(col("CustomerDOB").isNull()).count()
# temp_df.filter(size(col("dob_parts")) != 3).count()
# temp_df.filter(col("dob_parts").isNull()).select(size(col("dob_parts"))).show(5)
# temp_df.filter(col("dob_parts").isNull() | (size(col("dob_parts")) != 3)).count()

In [0]:
temp_df = temp_df.withColumn("dob_raw_year", element_at(col("dob_parts"),3))

In [0]:
temp_df.select("CustomerDOB","dob_parts", "dob_raw_year").show(5)

which rows have a 2-digit year (need the "19" prefix) versus the rare 4-digit ones like the 1/1/1800 row

In [0]:
temp_df = temp_df.withColumn("dob_year_lebgth", length(col("dob_raw_year")))

In [0]:
temp_df.select("dob_year_lebgth").distinct().show()

In [0]:
temp_df = temp_df.withColumn("dob_year_corrected", when(col("dob_year_lebgth")==2, concat(lit("19"),col("dob_raw_year"))).otherwise(col("dob_raw_year")))

In [0]:
temp_df.select("CustomerDOB", "dob_raw_year", "dob_year_lebgth", "dob_year_corrected").show(10)

In [0]:
temp_df.filter(col("dob_year_lebgth") == 4).select("CustomerDOB", "dob_raw_year", "dob_year_lebgth", "dob_year_corrected").show(10)

if using element_at indexing starts from 1, but if usinng [] array indexing it starts from 0

In [0]:
temp_df = temp_df.withColumn("CustomerDOB_corrected", concat(col("dob_parts")[0], lit("/"), col("dob_parts")[1], lit("/"), col("dob_year_corrected")))

In [0]:
temp_df.select("CustomerDOB", "CustomerDOB_corrected").show(10)

formatting customerDOB_corrected

In [0]:
temp_df = temp_df.withColumn("CustomerDOB_final", try_to_date(col("CustomerDOB_corrected"), "d/M/yyyy"))

In [0]:
temp_df.select("CustomerDOB", "CustomerDOB_corrected", "CustomerDOB_final").show(10)

In [0]:
# quality check for null
temp_df.filter(col("CustomerDOB_corrected").isNotNull() & col("CustomerDOB_final").isNull()).count()

> overwriting CustomerDOB with CustomerDOB_final, we have sucessfully updated customer_dob, so we can drop column from temp_df and assign temp_df to silver_bt

In [0]:
temp_df = temp_df.withColumn("CustomerDOB", col("CustomerDOB_final"))

checking whetehr values have been copied successfully

In [0]:
# temp_df.select("CustomerDOB", "CustomerDOB_final").show(10)

In [0]:
temp_df = temp_df.drop("CustomerDOB_corrected", "dob_parts", "dob_raw_year", "dob_year_corrected", "dob_year_lebgth", "CustomerDOB_final")

In [0]:
silver_bt = temp_df

In [0]:
# silver_bt.show(10)

Cleaning TrnasactionDate


I think its 2016 coz the all transactions cannot have same date as 16 across different year, also year cannot 2027, its still 2026
but given the date of birth someone born in 1983 cannot have a transcation date of 1916 so it is 2016 indeed! lets check the distinct value in year

Found it in discussion section of Kaggle project - _"The dataset is real, it was shared by a bank as part of a research project in 2016."_

In [0]:
# silver_bt.select("TransactionDate").distinct().show(30)

In [0]:
# temp_df2 = silver_bt.withColumn("txn_parts", split(col("TransactionDate"), "/"))
# temp_df2.select(element_at(col("txn_parts"), 3)).distinct().show()

In [0]:
# temp_df2 = temp_df2.withColumn("txn_year_corrected", concat(lit("20"), element_at(col("txn_parts"), 3)))

In [0]:
# Rebuild temp_df2 to clear any cached failed transformations
temp_df2 = silver_bt.withColumn("txn_parts", split(col("TransactionDate"), "/"))
temp_df2 = temp_df2.withColumn("txn_year_corrected", concat(lit("20"), element_at(col("txn_parts"), 3)))
temp_df2.select("TransactionDate", "txn_year_corrected").show(10)

In [0]:
temp_df2 = temp_df2.withColumn("transaction_date_final", concat(element_at(col("txn_parts"), 1), lit("/"), element_at(col("txn_parts"), 2), lit("/"), col("txn_year_corrected")))

In [0]:
# temp_df2.select("TransactionDate", "transaction_date_final").show(10)

finalizing column with correct date format

In [0]:
temp_df2 = temp_df2.withColumn("Transaction_date_parsed", try_to_date(col("transaction_date_final"), "d/M/yyyy") )

checking for null

In [0]:
temp_df2.filter(col("transaction_date_final").isNotNull() & col("Transaction_date_parsed").isNull()).count()

In [0]:
temp_df2 = temp_df2.withColumn("TransactionDate", col("Transaction_date_parsed"))
temp_df2 = temp_df2.drop("txn_parts", "Transaction_date_parsed", "txn_year_corrected", "transaction_date_final")
silver_bt = temp_df2

In [0]:
silver_bt.printSchema()

In [0]:
silver_bt.select("CustomerDOB", "TransactionDate"). show(10)

modifying other columns

In [0]:
silver_bt = silver_bt.drop("CustAccuntBalance")

In [0]:
silver_bt.printSchema()

casting cols

try_cast tolerates a future bad value gracefully (turns it into null, keeps running). Plain .cast() would throw a hard error and crash the entire job the moment even one future row has something genuinely unparseable in it

In [0]:
silver_bt = silver_bt.withColumn("CustAccountBalance", expr("try_cast(CustAccountBalance AS DOUBLE)"))

In [0]:
# silver_bt = silver_bt.withColumn("CustAccountBalance", col("CustAccountBalance").cast("double"))

transaction time col

In [0]:
silver_bt.select("TransactionTime").distinct().show(10)

In [0]:
silver_bt.select(length(col("TransactionTime").cast("string")).alias("len")).distinct().show()

What "padding" means: it's the general term for "add filler characters until something reaches a fixed length." lpad specifically means left-pad — add the filler characters onto the left side of the string. (There's also rpad, which pads on the right)

Why that's a real problem: if you tried to interpret 9 directly as HHMMSS, you'd misread it — is it 9:00:00? 0:00:09?

In [0]:
from pyspark.sql.functions import lpad

temp_df3 = silver_bt.withColumn(
    "time_padded",
    lpad(col("TransactionTime").cast("string"), 6, "0")
)

# Three arguments, each answering one question:

# col("TransactionTime").cast("string") — what to pad. Notice the .cast("string") — lpad only works on text, so the number has to become a string first (this is why 9 as a number can't be padded directly, but "9" as text can).
# 6 — the target length. You want every result to be exactly 6 characters, since that's how many digits HHMMSS needs.
# "0" — what character to pad with. You're filling the gap with zeros, since that's what a "missing leading zero" actually is.


temp_df3.select("TransactionTime", "time_padded").show(10)

lpad looks at whatever string you give it, checks its current length, and adds exactly as many pad characters as needed to reach your target — whether that's 1 missing character or 5.

In [0]:
temp_df3.filter(length(col("TransactionTime").cast("string")) < 4).select("TransactionTime", "time_padded").show(10)


In [0]:
# from pyspark.sql.functions import to_timestamp

# temp_df3 = temp_df3.withColumn("TransactionTime_parsed", to_timestamp(col("time_padded"), "HHmmss"))
# temp_df3.select("TransactionTime", "time_padded", "TransactionTime_parsed").show(10)

In [0]:
temp_df3.select("TransactionDate", "time_padded").show(10)

In [0]:
temp_df3 = temp_df3.withColumn("combinedstr", concat(col("TransactionDate").cast("string"), lit(" "), col("time_padded"))) 

In [0]:
temp_df3 = temp_df3.withColumn("TransactionDateTime", to_timestamp(col("combinedstr"), "yyyy-MM-dd HHmmss"))

In [0]:
temp_df3.select("combinedstr", "TransactionDateTime").show(10)

directly adding to silver_bt

In [0]:
silver_bt = silver_bt.withColumn(
    "TransactionDateTime",
    to_timestamp(
        concat_ws(" ", col("TransactionDate").cast("string"), lpad(col("TransactionTime").cast("string"), 6, "0")),
        "yyyy-MM-dd HHmmss"
    )
)

In [0]:
silver_bt.select("TransactionDate", "TransactionTime", "TransactionDateTime").show(5)

In [0]:
# silver_bt.select("CustGender").distinct().show()